In [1]:
!nvidia-smi

!python --version


Tue Dec  9 09:12:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
# Bạn có thể đổi tên folder nếu muốn
!mkdir -p "/content/drive/MyDrive/stylegan3_project"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/out_test"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/gen_raw"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/filtered/asian"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/filtered/child"


In [7]:


# kiểm tra conda chạy được
!/content/miniconda/bin/conda --version


/bin/bash: line 1: /content/miniconda/bin/conda: No such file or directory


In [8]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda --version


PREFIX=/content/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda
conda 25.9.1


In [9]:
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [10]:
!/content/miniconda/bin/conda create -y -n sg3 python=3.8
!/content/miniconda/bin/conda env list


Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ done
Channels:
 - defaults
Platform: linux-64
Solving environment: / done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /content/miniconda/envs/sg3

  added / updated specs:
    - python=3.8


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.12.2  |       h06a4308_0         125 KB
    pip-24.2                   |   py38h06a4308_0         2.2 MB
    python-3.8.20              |       he870216_0        23.8 MB
    setuptools-75.1.0          |   py38h06a4308_0         1.7 MB
    wheel-0.44.0               |   py38h06a4308_0         108 KB
    -----------------------------

In [11]:
# update pip
!/content/miniconda/bin/conda run -n sg3 python -m pip install -U pip

# deps cơ bản + các gói hay thiếu khi load network
!/content/miniconda/bin/conda run -n sg3 pip install ninja pyspng imageio-ffmpeg click scipy pillow numpy

# pytorch CUDA cho T4 (cu117)
!/content/miniconda/bin/conda run -n sg3 pip install \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

# kiểm tra GPU trong env
!/content/miniconda/bin/conda run -n sg3 python -c "import torch; print('torch',torch.__version__); print('cuda?',torch.cuda.is_available()); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 97.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 222.5 MB/s eta 0:00:00

Looking in indexes: https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 211.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typing-extensions to determine which version is compatible with other requirements. This could take a while.

torch 1.13.1+cu117
cuda? True
gpu Tesla T4



In [12]:
!rm -rf stylegan3
!git clone https://github.com/NVlabs/stylegan3.git

# cài requirements của repo
!/content/miniconda/bin/conda run -n sg3 pip install -r stylegan3/requirements.txt

# xoá cache extension để build sạch (tránh lỗi bias_act_plugin)
!rm -rf ~/.cache/torch_extensions


Cloning into 'stylegan3'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 212 (delta 99), reused 90 (delta 90), pack-reused 49 (from 1)
Receiving objects: 100% (212/212), 4.16 MiB | 17.68 MiB/s, done.
Resolving deltas: 100% (108/108), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'stylegan3/requirements.txt'

ERROR conda.cli.main_run:execute(127): `conda run pip install -r stylegan3/requirements.txt` failed. (See above for error)


In [28]:
!/content/miniconda/bin/conda run -n sg3 bash -lc 'cd stylegan3 && \
python gen_images.py \
  --outdir="/content/out_test" \
  --trunc=0.3 --seeds=6601-6601 \
  --network="https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"'


Loading networks from "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"...
Generating image for seed 6601 (0/1) ...
Setting up PyTorch plugin "bias_act_plugin"... Done.
Setting up PyTorch plugin "filtered_lrelu_plugin"... Done.



In [29]:
!/content/miniconda/bin/conda run -n sg3 bash -lc 'cd stylegan3 && \
python gen_images.py \
  --outdir="/content/gen_raw" \
  --trunc=0.3 --seeds=0-100 \
  --network="https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"'


Loading networks from "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"...
Generating image for seed 0 (0/101) ...
Setting up PyTorch plugin "bias_act_plugin"... Done.
Setting up PyTorch plugin "filtered_lrelu_plugin"... Done.
Generating image for seed 1 (1/101) ...
Generating image for seed 2 (2/101) ...
Generating image for seed 3 (3/101) ...
Generating image for seed 4 (4/101) ...
Generating image for seed 5 (5/101) ...
Generating image for seed 6 (6/101) ...
Generating image for seed 7 (7/101) ...
Generating image for seed 8 (8/101) ...
Generating image for seed 9 (9/101) ...
Generating image for seed 10 (10/101) ...
Generating image for seed 11 (11/101) ...
Generating image for seed 12 (12/101) ...
Generating image for seed 13 (13/101) ...
Generating image for seed 14 (14/101) ...
Generating image for seed 15 (15/101) ...
Generating image for seed 16 (16/101) ...
Generating image for seed 17 (17/101) ...
Generating ima

In [ ]:
#LỌC

In [15]:
!/content/miniconda/bin/conda run -n sg3 pip install -U insightface onnxruntime-gpu opencv-python transformers


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached cython-3.2.2-cp38-cp38-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.1/785.1 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.7 MB

In [ ]:
#Lọc và copy sang ổ Notebook

In [17]:
!pip install insightface==0.7.3 onnxruntime-gpu

  Using cached insightface-0.7.3.tar.gz (439 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 5.7 MB/s eta 0:00:00
Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 43.4 MB/s eta 0:00:00
Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071352 sha256=ce987e32c9650865718d5ebeaa0367339f833a9b5cf62d26f4c20760f7db0932
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


In [ ]:
!/content/miniconda/bin/conda run -n sg3 python - << "PY"
import os, re, csv, shutil, glob
import cv2

# ====== CONFIG ======
# Nếu bạn gen 10k test:
IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_raw/*.png"

# Nếu bạn gen 100k chunk:
#IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_100k/**/*.png"

FILTER_ASIAN = True
FILTER_CHILD = True
CHILD_MAX_AGE = 12  # đổi 18 nếu bạn muốn <18

# OUTPUT: copy về ổ Colab
OUT_ASIAN = "/content/filtered/asian"
OUT_CHILD = "/content/filtered/child"
CSV_PATH  = "/content/filtered/selected.csv"

os.makedirs(OUT_ASIAN, exist_ok=True)
os.makedirs(OUT_CHILD, exist_ok=True)
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# ====== seed parser ======
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ====== InsightFace (age) ======
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

# ====== CLIP race classifier (practical) ======
from transformers import CLIPProcessor, CLIPModel
import torch
import PIL.Image

model_id = "syntheticbot/clip-face-attribute-classifier"
clip_model = CLIPModel.from_pretrained(model_id)
clip_proc  = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

RACE_LABELS = ["White","Black","Indian","East Asian","Southeast Asian","Middle Eastern","Latino"]
ASIAN_SET = {"East Asian","Southeast Asian"}

def predict_race_bgr(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = PIL.Image.fromarray(img_rgb)
    inputs = clip_proc(text=RACE_LABELS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    idx = int(probs.argmax())
    return RACE_LABELS[idx], float(probs[idx])

# ====== Main loop ======
rows = []
kept_asian = kept_child = scanned = 0

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    faces = app.get(img)
    if not faces:
        continue

    # face lớn nhất
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))

    race, race_conf = predict_race_bgr(img)

    is_child = (age >= 0 and age <= CHILD_MAX_AGE) if FILTER_CHILD else False
    is_asian = (race in ASIAN_SET) if FILTER_ASIAN else False

    # Copy về ổ Colab
    if is_child:
        kept_child += 1
        shutil.copy2(path, os.path.join(OUT_CHILD, os.path.basename(path)))
    if is_asian:
        kept_asian += 1
        shutil.copy2(path, os.path.join(OUT_ASIAN, os.path.basename(path)))

    if is_child or is_asian:
        rows.append([os.path.basename(path), parse_seed(path), age, race, race_conf, int(is_asian), int(is_child)])

print("Scanned:", scanned)
print("Kept asian:", kept_asian, "->", OUT_ASIAN)
print("Kept child:", kept_child, "->", OUT_CHILD)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","race_pred","race_conf","is_asian","is_child"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)
PY


In [31]:
#Lọc 2
!cp -r /content/gen_raw/ /content/drive/MyDrive/

In [30]:
!/content/miniconda/bin/conda run -n sg3 python - << "PY"
import os, re, csv, shutil, glob
import cv2

# ================== INPUT (Drive) ==================
# Nếu bạn gen 100k theo chunk:
IN_GLOB = "/content/gen_raw/**/*.png"
# Nếu bạn gen 1 folder:
# IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_raw/*.png"

# ================== THRESHOLDS ==================
CHILD_MAX_AGE   = 8
ADULT_MIN_AGE   = 18
ELDERLY_MIN_AGE = 65

# ================== OUTPUT (Colab disk) ==================
OUTROOT  = "/content/filtered_asian"
OUT_CHILD = os.path.join(OUTROOT, "CHILD8")
OUT_ELD   = os.path.join(OUTROOT, "Elderly")
OUT_MALE  = os.path.join(OUTROOT, "Male")
OUT_FEMA  = os.path.join(OUTROOT, "Female")
CSV_PATH  = os.path.join(OUTROOT, "selected_asian.csv")

for d in [OUT_CHILD, OUT_ELD, OUT_MALE, OUT_FEMA]:
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# ================== seed parser ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ================== InsightFace (age+gender) ==================
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

def infer_age_gender(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    # chọn face lớn nhất
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))  # thường: 0=female, 1=male
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

# ================== CLIP race classifier (Asian gate) ==================
from transformers import CLIPProcessor, CLIPModel
import torch
import PIL.Image

model_id = "syntheticbot/clip-face-attribute-classifier"
clip_model = CLIPModel.from_pretrained(model_id)
clip_proc  = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

RACE_LABELS = ["White","Black","Indian","East Asian","Southeast Asian","Middle Eastern","Latino"]
ASIAN_SET = {"East Asian","Southeast Asian"}

def predict_race_bgr(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = PIL.Image.fromarray(img_rgb)
    inputs = clip_proc(text=RACE_LABELS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    idx = int(probs.argmax())
    return RACE_LABELS[idx], float(probs[idx])

# ================== MAIN ==================
scanned = 0
asian_pass = 0
kept = {"CHILD8":0, "Elderly":0, "MaleAdult":0, "FemaleAdult":0}
rows = []

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    # Gate: Asian?
    race, race_conf = predict_race_bgr(img)
    if race not in ASIAN_SET:
        continue
    asian_pass += 1

    ag = infer_age_gender(img)
    if ag is None:
        continue
    age, gender = ag

    base = os.path.basename(path)
    seed = parse_seed(path)

    # Bucket rules (ưu tiên CHILD8/Elderly trước; adult mới chia gender)
    bucket = None
    if age >= 0 and age <= CHILD_MAX_AGE:
        bucket = "CHILD8"
        dst = os.path.join(OUT_CHILD, base)
        kept["CHILD8"] += 1
        shutil.copy2(path, dst)
    elif age >= ELDERLY_MIN_AGE:
        bucket = "Elderly"
        dst = os.path.join(OUT_ELD, base)
        kept["Elderly"] += 1
        shutil.copy2(path, dst)
    elif age >= ADULT_MIN_AGE and age < ELDERLY_MIN_AGE:
        # Adult: chia gender
        if gender == "M":
            bucket = "Male"
            dst = os.path.join(OUT_MALE, base)
            kept["MaleAdult"] += 1
            shutil.copy2(path, dst)
        elif gender == "F":
            bucket = "Female"
            dst = os.path.join(OUT_FEMA, base)
            kept["FemaleAdult"] += 1
            shutil.copy2(path, dst)
        else:
            # gender unknown -> bỏ qua hoặc bạn có thể tạo folder "AdultUnknown"
            bucket = "AdultUnknown"
            dst = None
    else:
        bucket = "Other"  # ví dụ 9-17, hoặc age=-1
        dst = None

    # Ghi CSV cho mọi ảnh Asian pass (kể cả không bucket) để bạn audit
    rows.append([base, seed, age, gender, race, race_conf, bucket, dst])

print("Scanned:", scanned)
print("Asian passed:", asian_pass)
print("Kept:", kept)
print("Output root:", OUTROOT)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","gender","race_pred","race_conf","bucket","dst_path"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)
PY


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied provi

Some weights of the model checkpoint at syntheticbot/clip-face-attribute-classifier were not used when initializing CLIPModel: ['age_head.bias', 'age_head.weight', 'gender_head.bias', 'gender_head.weight', 'race_head.bias', 'race_head.weight', 'vision_model.vision_model.embeddings.class_embedding', 'vision_model.vision_model.embeddings.patch_embedding.weight', 'vision_model.vision_model.embeddings.position_embedding.weight', 'vision_model.vision_model.encoder.layers.0.layer_norm1.bias', 'vision_model.vision_model.encoder.layers.0.layer_norm1.weight', 'vision_model.vision_model.encoder.layers.0.layer_norm2.bias', 'vision_model.vision_model.encoder.layers.0.layer_norm2.weight', 'vision_model.vision_model.encoder.layers.0.mlp.fc1.bias', 'vision_model.vision_model.encoder.layers.0.mlp.fc1.weight', 'vision_model.vision_model.encoder.layers.0.mlp.fc2.bias', 'vision_model.vision_model.encoder.layers.0.mlp.fc2.weight', 'vision_model.vision_model.encoder.layers.0.self_attn.k_proj.bias', 'vision

Scanned: 101
Asian passed: 0
Kept: {'CHILD8': 0, 'Elderly': 0, 'MaleAdult': 0, 'FemaleAdult': 0}
Output root: /content/filtered_asian
Wrote: /content/filtered_asian/selected_asian.csv


NameError: name 'PY' is not defined